# Comparison of multiple mechanical and ultrasonic experiments

This notebook processes and compares any number of experiments produced by the mechanical-reduction notebook and **Mech Wave Analyzer v3**.

For every experiment it:

1. imports one reduced mechanical file and one synchronized waveform-pick file;
2. verifies the exported acoustic time and mechanical row coordinates;
3. calculates P- and S-wave flight times, \(V_P\), \(V_S\), and \(V_P/V_S\);
4. interpolates mechanical variables at every waveform time;
5. overlays all experiments in one interactive comparison window.

The plotting window uses one color per experiment, compact drop-down selectors, a mechanical axis on the left, and a dynamically scaled acoustic axis on the right.

## 0. Environment

Use the same Conda environment as the waveform analyzer:

```bash
conda activate mechwave
python -m pip install numpy pandas matplotlib PyQt6 jupyter
```

The final cell opens a separate Qt window. Close it when finished and re-run the cell to reopen it.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

COLORS = [
    '#167C80', '#D9792B', '#2B5C8F', '#C73E3A', '#7A5195',
    '#2E7D32', '#8C564B', '#E377C2', '#6B7280', '#BCBD22',
]
INK = '#263238'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

## 1. Define the experiments

Add one dictionary for each experiment. The `name` values must be unique.

Each experiment can have its own:

- arrival-picking method;
- cross-correlation acceptance threshold;
- P and S propagation lengths;
- independently measured P and S system delays;
- decision about axial-shortening correction.

`arrival_method` may be `cross_correlation`, `sta_lta`, `manual`, or `best_available`. For final comparisons, use the same picking method and quality criterion whenever possible.

The 50 mm values below match the example reduction, but the actual transmitter-to-receiver path must be verified for every specimen.

In [ ]:
EXPERIMENTS = [
    {
        'name': 'Sandstone example',
        'mechanical_file': 'triax_try_sandstone_acousto_mech.txt',
        'acoustic_file': 'synched_waveform_picks.csv',
        'arrival_method': 'cross_correlation',
        'minimum_cc': 0.50,
        'p_path_length_mm': 50.0,
        's_path_length_mm': 50.0,
        'p_system_delay_us': 0.0,
        's_system_delay_us': 0.0,
        'apply_axial_shortening': True,
        'shortening_column': 'axial_shortening_corrected_mm',
    },

    # Copy and edit this block for every additional experiment:
    # {
    #     'name': 'Granite example',
    #     'mechanical_file': 'granite_acousto_mech.txt',
    #     'acoustic_file': 'granite_synched_waveform_picks.csv',
    #     'arrival_method': 'cross_correlation',
    #     'minimum_cc': 0.50,
    #     'p_path_length_mm': 50.0,
    #     's_path_length_mm': 50.0,
    #     'p_system_delay_us': 0.0,
    #     's_system_delay_us': 0.0,
    #     'apply_axial_shortening': True,
    #     'shortening_column': 'axial_shortening_corrected_mm',
    # },
]

# Keep True in normal use. Automated validation can set the environment variable
# MECH_ACOUSTIC_HEADLESS=1 to skip the Qt window.
OPEN_POPUP = os.environ.get('MECH_ACOUSTIC_HEADLESS', '0') != '1'

## 2. Import, synchronization, and velocity functions

The acoustic export contains both `Mech_Time_s` and `Mech_Row_Index`. Time is used for interpolation because it preserves the fractional position of each waveform. The rounded mechanical row is used as an independent synchronization audit.

Velocity is calculated from corrected flight time:

\[
t=t_{pick}-t_{system}, \qquad
V\,[\mathrm{m\,s^{-1}}]=1000\frac{L\,[\mathrm{mm}]}{t\,[\mu\mathrm{s}]}
\]

When propagation is axial and shortening is positive in compression:

\[
L_i=L_0-\Delta L_i
\]

The calculation is performed separately for every experiment before comparison.

In [ ]:
PICK_COLUMNS = {
    'manual': ('P_Manual_us', 'S_Manual_us'),
    'sta_lta': ('P_Auto_us', 'S_Auto_us'),
    'cross_correlation': ('P_CC_us', 'S_CC_us'),
}

REQUIRED_MECHANICAL = {'time_s', 'record'}
REQUIRED_ACOUSTIC = {
    'Interval_Num', 'File_Name', 'Trace_Index', 'Mech_Time_s', 'Mech_Row_Index',
    'P_Manual_us', 'S_Manual_us', 'P_Auto_us', 'S_Auto_us',
    'P_CC_us', 'S_CC_us', 'P_CC_Coefficient', 'S_CC_Coefficient',
}


def locate_file(filename):
    """Find an input file in the notebook folder or common nearby folders."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / 'upload' / filename,
        Path.cwd().parent / filename,
        Path.cwd().parent / 'upload' / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = '\n'.join(f'  - {candidate}' for candidate in candidates)
    raise FileNotFoundError(f'Could not find {filename}. Searched:\n{searched}')


def read_mechanical_output(path):
    """Read the tab-separated mechanical output and preserve its units row."""
    with open(path, 'r', encoding='utf-8-sig') as stream:
        header = stream.readline().rstrip('\n\r').split('\t')
        unit_values = stream.readline().rstrip('\n\r').split('\t')
    unit_values += [''] * (len(header) - len(unit_values))
    units = dict(zip(header, unit_values[:len(header)]))

    frame = pd.read_csv(path, sep='\t', skiprows=[1], encoding='utf-8-sig')
    for column in frame.columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame, units


def interpolate_mechanical(frame, column, times_s):
    """Interpolate a numerical mechanical variable at waveform times."""
    if column not in frame.columns:
        raise KeyError(f'Mechanical column not found: {column}')
    time = frame['time_s'].to_numpy(float)
    values = pd.to_numeric(frame[column], errors='coerce').to_numpy(float)
    valid = np.isfinite(time) & np.isfinite(values)
    if valid.sum() < 2:
        return np.full(len(times_s), np.nan)
    return np.interp(np.asarray(times_s, dtype=float), time[valid], values[valid])


def first_available_with_method(frame, phase):
    """Select manual, then cross-correlation, then STA/LTA for one phase."""
    candidates = [
        ('manual', f'{phase}_Manual_us'),
        ('cross_correlation', f'{phase}_CC_us'),
        ('sta_lta', f'{phase}_Auto_us'),
    ]
    values = pd.Series(np.nan, index=frame.index, dtype=float)
    methods = pd.Series(pd.NA, index=frame.index, dtype='object')
    for method, column in candidates:
        available = values.isna() & frame[column].notna()
        values.loc[available] = frame.loc[available, column]
        methods.loc[available] = method
    return values, methods


def select_arrivals(frame, settings):
    """Add the selected P and S arrival series without deleting original picks."""
    result = frame.copy()
    method = settings['arrival_method']
    if method == 'best_available':
        result['P_pick_us'], result['P_pick_method'] = first_available_with_method(result, 'P')
        result['S_pick_us'], result['S_pick_method'] = first_available_with_method(result, 'S')
    elif method in PICK_COLUMNS:
        p_column, s_column = PICK_COLUMNS[method]
        result['P_pick_us'] = result[p_column]
        result['S_pick_us'] = result[s_column]
        result['P_pick_method'] = method
        result['S_pick_method'] = method
    else:
        raise ValueError(f"Unknown arrival_method for {settings['name']}: {method!r}")

    # Enforce the selected CC threshold wherever a pick originates from CC.
    p_cc = result['P_pick_method'].eq('cross_correlation')
    s_cc = result['S_pick_method'].eq('cross_correlation')
    result.loc[p_cc & (result['P_CC_Coefficient'] < settings['minimum_cc']), 'P_pick_us'] = np.nan
    result.loc[s_cc & (result['S_CC_Coefficient'] < settings['minimum_cc']), 'S_pick_us'] = np.nan
    return result


def process_experiment(settings):
    """Load, audit, synchronize, and calculate acoustic properties for one experiment."""
    mechanical_path = locate_file(settings['mechanical_file'])
    acoustic_path = locate_file(settings['acoustic_file'])
    mechanical, units = read_mechanical_output(mechanical_path)
    acoustic = pd.read_csv(acoustic_path)

    missing_mechanical = REQUIRED_MECHANICAL.difference(mechanical.columns)
    missing_acoustic = REQUIRED_ACOUSTIC.difference(acoustic.columns)
    if missing_mechanical:
        raise KeyError(f"{settings['name']} mechanical columns missing: {sorted(missing_mechanical)}")
    if missing_acoustic:
        raise KeyError(f"{settings['name']} acoustic columns missing: {sorted(missing_acoustic)}")
    if not mechanical['time_s'].is_monotonic_increasing:
        raise ValueError(f"{settings['name']} mechanical time_s is not monotonic.")

    for column in acoustic.columns:
        if column != 'File_Name':
            acoustic[column] = pd.to_numeric(acoustic[column], errors='coerce')

    row_indices = acoustic['Mech_Row_Index'].round().astype('Int64')
    rows_valid = row_indices.between(0, len(mechanical) - 1).fillna(False)
    if not rows_valid.all():
        raise IndexError(f"{settings['name']} has waveform rows outside the mechanical table.")
    if not acoustic['Mech_Time_s'].between(
        mechanical['time_s'].min(), mechanical['time_s'].max()
    ).all():
        raise ValueError(f"{settings['name']} has waveform times outside the mechanical record.")

    row_times = mechanical.iloc[row_indices.astype(int).to_numpy()]['time_s'].to_numpy(float)
    mismatch = np.abs(row_times - acoustic['Mech_Time_s'].to_numpy(float))
    mechanical_dt = float(np.nanmedian(np.diff(mechanical['time_s'].to_numpy(float))))

    acoustic = select_arrivals(acoustic, settings)
    acoustic['P_flight_time_us'] = acoustic['P_pick_us'] - float(settings['p_system_delay_us'])
    acoustic['S_flight_time_us'] = acoustic['S_pick_us'] - float(settings['s_system_delay_us'])
    acoustic.loc[acoustic['P_flight_time_us'] <= 0, 'P_flight_time_us'] = np.nan
    acoustic.loc[acoustic['S_flight_time_us'] <= 0, 'S_flight_time_us'] = np.nan

    if settings['apply_axial_shortening']:
        shortening = interpolate_mechanical(
            mechanical, settings['shortening_column'], acoustic['Mech_Time_s']
        )
    else:
        shortening = np.zeros(len(acoustic), dtype=float)
    acoustic['path_shortening_mm'] = shortening
    acoustic['P_path_length_mm'] = float(settings['p_path_length_mm']) - shortening
    acoustic['S_path_length_mm'] = float(settings['s_path_length_mm']) - shortening
    if ((acoustic['P_path_length_mm'] <= 0) | (acoustic['S_path_length_mm'] <= 0)).any():
        raise ValueError(f"{settings['name']} has a non-positive calculated path length.")

    acoustic['Vp_m_per_s'] = 1000.0 * acoustic['P_path_length_mm'] / acoustic['P_flight_time_us']
    acoustic['Vs_m_per_s'] = 1000.0 * acoustic['S_path_length_mm'] / acoustic['S_flight_time_us']
    acoustic['Vp_over_Vs'] = acoustic['Vp_m_per_s'] / acoustic['Vs_m_per_s']

    return {
        'name': settings['name'],
        'settings': settings.copy(),
        'mechanical': mechanical,
        'mechanical_units': units,
        'acoustic': acoustic,
        'mechanical_path': mechanical_path,
        'acoustic_path': acoustic_path,
        'mechanical_dt_s': mechanical_dt,
        'maximum_row_time_mismatch_s': float(np.nanmax(mismatch)),
        'synchronization_passed': bool(np.nanmax(mismatch) <= 0.5 * mechanical_dt + 1e-9),
    }

## 3. Process all experiments

The summary reports file sizes, synchronization quality, and the number of valid acoustic results for every experiment. Resolve warnings before comparing curves.

In [ ]:
names = [item['name'] for item in EXPERIMENTS]
if len(names) != len(set(names)):
    raise ValueError('Every experiment name must be unique.')
if not EXPERIMENTS:
    raise ValueError('Add at least one experiment to EXPERIMENTS.')

experiment_results = [process_experiment(settings) for settings in EXPERIMENTS]

summary_rows = []
for result in experiment_results:
    acoustic = result['acoustic']
    summary_rows.append({
        'experiment': result['name'],
        'mechanical_rows': len(result['mechanical']),
        'waveform_traces': len(acoustic),
        'arrival_method': result['settings']['arrival_method'],
        'maximum_row_time_mismatch_s': result['maximum_row_time_mismatch_s'],
        'half_sample_tolerance_s': 0.5 * result['mechanical_dt_s'],
        'sync_passed': result['synchronization_passed'],
        'valid_Vp': int(acoustic['Vp_m_per_s'].notna().sum()),
        'valid_Vs': int(acoustic['Vs_m_per_s'].notna().sum()),
        'valid_Vp_over_Vs': int(acoustic['Vp_over_Vs'].notna().sum()),
    })

comparison_summary = pd.DataFrame(summary_rows)
display(comparison_summary)

if not comparison_summary['sync_passed'].all():
    print('WARNING: at least one experiment failed the time-versus-row synchronization audit.')
else:
    print('All experiments passed the synchronization audit.')

## 4. Interactive comparison plot

The pop-up window overlays every experiment.

- **X variable** and **Mechanical Y** are selected from the numerical columns shared by all mechanical files.
- **Acoustic right axis** selects flight time, velocity, or `Vp/Vs`; the right-axis scale and unit update automatically.
- One color identifies each experiment across both axes.
- Solid lines represent mechanical curves. Acoustic P uses circles, S uses squares, and `Vp/Vs` uses diamonds.
- Each waveform file is drawn as a separate acoustic segment, so acquisition gaps are not connected.
- Hover over any curve to identify the experiment and read its coordinates.

For the most direct comparison between specimens of different duration, useful choices are axial strain on X, axial stress on the left, and velocity or `Vp/Vs` on the right.

In [ ]:
ACOUSTIC_DISPLAYS = {
    'P flight time': (['P_flight_time_us'], 'Flight time [µs]'),
    'S flight time': (['S_flight_time_us'], 'Flight time [µs]'),
    'P and S flight times': (['P_flight_time_us', 'S_flight_time_us'], 'Flight time [µs]'),
    'Vp': (['Vp_m_per_s'], 'Velocity [m/s]'),
    'Vs': (['Vs_m_per_s'], 'Velocity [m/s]'),
    'Vp and Vs': (['Vp_m_per_s', 'Vs_m_per_s'], 'Velocity [m/s]'),
    'Vp/Vs ratio': (['Vp_over_Vs'], 'Vp/Vs [–]'),
}

PHASE_STYLE = {
    'P_flight_time_us': ('P flight time', 'o', '-'),
    'S_flight_time_us': ('S flight time', 's', '--'),
    'Vp_m_per_s': ('Vp', 'o', '-'),
    'Vs_m_per_s': ('Vs', 's', '--'),
    'Vp_over_Vs': ('Vp/Vs', 'D', '-'),
}


def shared_numeric_columns(results):
    """Return numerical, non-raw mechanical columns present in every experiment."""
    common = None
    first_order = []
    for result in results:
        columns = [
            column for column in result['mechanical'].columns
            if pd.api.types.is_numeric_dtype(result['mechanical'][column])
            and not column.endswith('_raw')
        ]
        if common is None:
            common = set(columns)
            first_order = columns
        else:
            common &= set(columns)
    return [column for column in first_order if column in common]


def open_multi_experiment_explorer(results):
    """Open a Qt comparison window and overlay all processed experiments."""
    from PyQt6.QtWidgets import (
        QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
        QLabel, QComboBox, QPushButton,
    )
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
    from matplotlib.backends.backend_qtagg import NavigationToolbar2QT as NavigationToolbar

    numeric_columns = shared_numeric_columns(results)
    if len(numeric_columns) < 2:
        raise ValueError('The experiments do not share at least two numerical mechanical columns.')
    preferred_x = 'axial_strain_percent' if 'axial_strain_percent' in numeric_columns else (
        'time_s' if 'time_s' in numeric_columns else numeric_columns[0]
    )
    preferred_y = 'axial_stress_MPa' if 'axial_stress_MPa' in numeric_columns else numeric_columns[1]
    preferred_acoustic = 'Vp and Vs'

    app = QApplication.instance()
    if app is None:
        app = QApplication([])
    window = QMainWindow()
    window.setWindowTitle('Multiple-experiment mechanical and acoustic comparison')
    window.resize(1500, 880)

    central = QWidget()
    main_layout = QVBoxLayout(central)
    main_layout.setContentsMargins(8, 8, 8, 8)
    controls = QHBoxLayout()

    def add_combo(label_text, values, selected, width):
        controls.addWidget(QLabel(label_text))
        combo = QComboBox()
        combo.addItems(values)
        combo.setCurrentText(selected)
        combo.setMinimumWidth(width)
        controls.addWidget(combo)
        return combo

    x_combo = add_combo('X variable:', numeric_columns, preferred_x, 235)
    y_combo = add_combo('Mechanical Y:', numeric_columns, preferred_y, 255)
    acoustic_combo = add_combo(
        'Acoustic right axis:', list(ACOUSTIC_DISPLAYS), preferred_acoustic, 215
    )
    reset_button = QPushButton('Reset view')
    reset_button.setMinimumWidth(105)
    controls.addSpacing(8)
    controls.addWidget(reset_button)
    controls.addStretch()

    figure = Figure(figsize=(12.8, 7.2))
    canvas = FigureCanvas(figure)
    toolbar = NavigationToolbar(canvas, window)
    left_axis = figure.add_subplot(111)
    right_axis = left_axis.twinx()
    main_layout.addLayout(controls)
    main_layout.addWidget(toolbar)
    main_layout.addWidget(canvas, stretch=1)
    window.setCentralWidget(central)

    state = {
        'x': preferred_x,
        'y': preferred_y,
        'acoustic': preferred_acoustic,
        'artists': [],
        'annotations': [],
    }

    def unit_for(column):
        units = [result['mechanical_units'].get(column, '') for result in results]
        unique = {unit for unit in units if unit}
        return next(iter(unique)) if len(unique) == 1 else ''

    def mechanical_label(column):
        unit = unit_for(column)
        return f'{column} [{unit}]' if unit else column

    def acoustic_x(result, column):
        acoustic = result['acoustic']
        if column == 'time_s':
            return acoustic['Mech_Time_s'].to_numpy(float)
        return interpolate_mechanical(
            result['mechanical'], column, acoustic['Mech_Time_s']
        )

    def redraw():
        left_axis.clear()
        right_axis.clear()
        state['artists'] = []
        state['annotations'] = []
        acoustic_columns, acoustic_ylabel = ACOUSTIC_DISPLAYS[state['acoustic']]

        for experiment_index, result in enumerate(results):
            color = COLORS[experiment_index % len(COLORS)]
            mechanical = result['mechanical']
            acoustic = result['acoustic']
            name = result['name']

            mechanical_line, = left_axis.plot(
                mechanical[state['x']], mechanical[state['y']],
                color=color, lw=1.7, alpha=0.92,
                label=f'{name} — {state["y"]}', zorder=2
            )
            mechanical_line.set_picker(5)
            state['artists'].append((mechanical_line, left_axis, f'{name} — mechanical'))

            x_values_all = acoustic_x(result, state['x'])
            for acoustic_column in acoustic_columns:
                phase_label, marker, line_style = PHASE_STYLE[acoustic_column]
                first_segment = True
                for _, group in acoustic.groupby(['Interval_Num', 'File_Name'], sort=False):
                    indices = group.index.to_numpy()
                    x_values = x_values_all[indices]
                    y_values = acoustic.loc[indices, acoustic_column].to_numpy(float)
                    valid = np.isfinite(x_values) & np.isfinite(y_values)
                    if not valid.any():
                        continue
                    line, = right_axis.plot(
                        x_values[valid], y_values[valid],
                        color=color, linestyle=line_style, marker=marker,
                        markersize=3.7, lw=1.15, alpha=0.88,
                        label=f'{name} — {phase_label}' if first_segment else '_nolegend_',
                        zorder=3,
                    )
                    line.set_picker(5)
                    state['artists'].append((line, right_axis, f'{name} — {phase_label}'))
                    first_segment = False

        left_axis.set_xlabel(mechanical_label(state['x']))
        left_axis.set_ylabel(mechanical_label(state['y']), color='#167C80')
        right_axis.set_ylabel(acoustic_ylabel, color='#D9792B')
        left_axis.yaxis.set_label_position('left')
        left_axis.yaxis.tick_left()
        right_axis.yaxis.set_label_position('right')
        right_axis.yaxis.tick_right()
        left_axis.tick_params(axis='y', colors='#167C80')
        right_axis.tick_params(axis='y', colors='#D9792B')
        left_axis.spines['left'].set_color('#167C80')
        right_axis.spines['right'].set_color('#D9792B')
        right_axis.spines['right'].set_visible(True)
        left_axis.grid(alpha=0.22)
        right_axis.relim()
        right_axis.autoscale_view()
        left_axis.set_title(
            f"Multiple experiments — {state['y']} and {state['acoustic']}",
            color=INK, pad=34
        )

        handles_left, labels_left = left_axis.get_legend_handles_labels()
        handles_right, labels_right = right_axis.get_legend_handles_labels()
        left_axis.legend(
            handles_left + handles_right, labels_left + labels_right,
            loc='upper center', bbox_to_anchor=(0.5, 1.035),
            ncol=min(4, max(1, len(labels_left + labels_right))),
            fontsize=8.5, frameon=True
        )

        for axis, edge_color in ((left_axis, '#167C80'), (right_axis, '#D9792B')):
            annotation = axis.annotate(
                '', xy=(0, 0), xytext=(14, 14), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                          edgecolor=edge_color, alpha=0.96),
                arrowprops=dict(arrowstyle='->', color=edge_color), zorder=10
            )
            annotation.set_visible(False)
            state['annotations'].append(annotation)

        figure.subplots_adjust(left=0.085, right=0.90, bottom=0.11, top=0.86)
        canvas.draw_idle()

    def choose_x(value):
        state['x'] = value
        redraw()

    def choose_y(value):
        state['y'] = value
        redraw()

    def choose_acoustic(value):
        state['acoustic'] = value
        redraw()

    def reset_view(_checked=False):
        left_axis.relim()
        left_axis.autoscale_view()
        right_axis.relim()
        right_axis.autoscale_view()
        for annotation in state['annotations']:
            annotation.set_visible(False)
        canvas.draw_idle()

    def hover(event):
        was_visible = any(annotation.get_visible() for annotation in state['annotations'])
        for annotation in state['annotations']:
            annotation.set_visible(False)
        found = False
        for artist, axis, label in reversed(state['artists']):
            contains, information = artist.contains(event)
            indices = information.get('ind', []) if contains else []
            if not len(indices):
                continue
            point_index = int(indices[0])
            x_value = np.asarray(artist.get_xdata())[point_index]
            y_value = np.asarray(artist.get_ydata())[point_index]
            annotation = state['annotations'][0 if axis is left_axis else 1]
            annotation.xy = (x_value, y_value)
            annotation.set_text(f'{label}\nx: {x_value:.6g}\ny: {y_value:.6g}')
            annotation.set_visible(True)
            found = True
            break
        if found or was_visible:
            canvas.draw_idle()

    x_combo.currentTextChanged.connect(choose_x)
    y_combo.currentTextChanged.connect(choose_y)
    acoustic_combo.currentTextChanged.connect(choose_acoustic)
    reset_button.clicked.connect(reset_view)
    hover_connection = canvas.mpl_connect('motion_notify_event', hover)
    redraw()
    window.show()

    window._comparison_objects = (
        app, figure, canvas, left_axis, right_axis, x_combo, y_combo,
        acoustic_combo, reset_button, hover_connection,
    )
    return window


if OPEN_POPUP:
    get_ipython().run_line_magic('matplotlib', 'qt')
    comparison_window = open_multi_experiment_explorer(experiment_results)
else:
    print('Headless mode: comparison window was not opened.')

## Result objects

- `experiment_results` is a list containing the settings, mechanical DataFrame, acoustic-analysis DataFrame, units, file paths, and synchronization audit for each experiment.
- `comparison_summary` provides a compact quality-control table.

The notebook intentionally does not smooth, normalize, or resample the measured curves. Those decisions should be made only after the overlaid experiments and their acquisition coverage have been inspected.